# Lab 06 — Erasure Coding in Practice

Erasure Coding (EC) is the technology that lets distributed storage systems achieve
high durability with far less overhead than traditional replication.

### The core idea: Reed-Solomon codes

**RS(k, m)** splits an object into **k data shards** and computes **m parity shards**
using Galois Field arithmetic. Any **k** shards (out of k+m total) are sufficient
to reconstruct the original data — even if up to **m** shards are lost.

```
Original data  ──RS(4,2)──▶  [D1][D2][D3][D4]  ←  4 data shards
                              [P1][P2]           ←  2 parity shards
                               ↓   ↓   ↓   ↓
                            drive0 1  2  3      ←  distributed across drives

If drive1 & drive2 fail: [D1][xx][xx][D4][P1][P2] → still 4 shards → reconstruct D2,D3
```

In this lab you'll upload objects of various sizes, inspect their metadata,
compare storage efficiency across EC schemes, and observe the cluster health.


---
## 0 · Prerequisites

1. RustFS is running: `docker compose up -d`
2. Virtual environment active with `boto3` installed: `uv sync`
3. Docker CLI available in PATH (for container status checks)


---
## Setup — Connect and Create Bucket


In [ ]:
import boto3
import subprocess
import math
from botocore.config import Config

s3 = boto3.client(
    service_name='s3',
    endpoint_url='http://localhost:9000',
    aws_access_key_id='admin',
    aws_secret_access_key='adminpassword',
    region_name='us-east-1',
    use_ssl=False,
    config=Config(
        signature_version='s3v4',
        s3={'addressing_style': 'path'},
    ),
)

BUCKET = 'lab6-ec'

existing = {b['Name'] for b in s3.list_buckets()['Buckets']}
if BUCKET not in existing:
    s3.create_bucket(Bucket=BUCKET)
    print(f'✅ Created bucket: {BUCKET}')
else:
    print(f'✅ Bucket already exists: {BUCKET}')


---
## 1 · Verify Cluster Health

Before uploading, confirm RustFS is healthy and all drives are online.
The liveness endpoint returns HTTP 200 when the server is ready.


In [ ]:
# Liveness check
result = subprocess.run(
    ['curl', '-sf', '-o', '/dev/null', '-w', '%{http_code}',
     'http://localhost:9000/health'],
    capture_output=True, text=True,
)
code = result.stdout.strip()
print(f'RustFS liveness: HTTP {code} — {"healthy ✅" if code == "200" else f"unexpected status ⚠️"}')

# Container status
result = subprocess.run(
    ['docker', 'compose', 'ps'],
    capture_output=True, text=True, cwd='..'
)
print('\nContainer status:')
print(result.stdout or '(no output — run from repo root)')

---
## 2 · Upload Objects of Varying Sizes

We upload objects ranging from 1 KB to 10 MB to observe how RustFS handles
different sizes. Internally, shard size is dynamically adjusted (64 KB – 4 MB)
based on object size.


In [ ]:
# Object sizes to test: key → size in bytes
TEST_OBJECTS = {
    'ec/object_1KB.bin':   1 * 1024,
    'ec/object_10KB.bin':  10 * 1024,
    'ec/object_100KB.bin': 100 * 1024,
    'ec/object_1MB.bin':   1 * 1024 * 1024,
    'ec/object_10MB.bin':  10 * 1024 * 1024,
}

for key, size in TEST_OBJECTS.items():
    content = b'a' * size
    s3.put_object(Bucket=BUCKET, Key=key, Body=content)
    print(f'  Uploaded {key:30s}  ({size:>12,} bytes)')

print(f'\n✅ {len(TEST_OBJECTS)} objects uploaded.')


---
## 3 · Inspect Object Metadata

`head_object` returns the **ETag** — a content fingerprint computed by S3.
For multipart uploads the ETag includes the number of parts; for single-part
uploads it is the MD5 of the content.

Note that `ContentLength` reflects the **logical object size** as the client sees it —
not the raw storage used (which is higher due to EC parity shards).


In [ ]:
print(f'  {"Key":30s}  {"Logical size":>14}  ETag')
print('  ' + '-' * 80)
for key in TEST_OBJECTS:
    meta = s3.head_object(Bucket=BUCKET, Key=key)
    size = meta['ContentLength']
    etag = meta['ETag']
    print(f'  {key:30s}  {size:>14,} bytes  {etag}')


---
## 4 · Storage Efficiency Comparison

**Storage efficiency** = data shards / total shards = `k / (k + m)`.

Higher efficiency means less overhead — but fewer drives can fail before data loss.
The right trade-off depends on your cluster size and durability requirements.


In [ ]:
# EC scheme comparison
# (name, data_shards k, parity_shards m, notes)
schemes = [
    ('3x Replication',      1,  2, 'Simple; any 1 of 3 copies survives'),
    ('RS(4,2)  — EC:2',     4,  2, 'RustFS default; 67% efficiency'),
    ('RS(4,4)  — EC:4',     4,  4, 'High fault tolerance; 50% efficiency'),
    ('RS(6,2)',              6,  2, 'High efficiency; 75%'),
    ('RS(10,4)',            10,  4, 'Large clusters; 71%'),
]

header = f'{"Scheme":<22}  {"k":>2}  {"m":>2}  {"n":>2}  {"Efficiency":>11}  {"Max failures":>12}'
print(header)
print('-' * len(header))
for name, k, m, note in schemes:
    n = k + m
    eff = k / n
    marker = '  ← this lab' if k == 4 and m == 2 else ''
    print(f'{name:<22}  {k:>2}  {m:>2}  {n:>2}  {eff:>10.1%}  {m:>12}{marker}')

print()
print('Interpretation: RS(4,2) stores a 10 GB dataset in 15 GB (10/0.67).')
print('3x replication stores the same dataset in 30 GB — 2× less efficient.')


---
## 5 · Simulating a Drive Failure

The reliable way to simulate a failure is to **rename the bucket directory** on the
target drive. RustFS detects `ENOENT` on the first `GET` and reconstructs the object
from surviving shards — completely transparent to the S3 client.

> **Why not `chmod 000`?** The process may keep the directory open via a cached file
> descriptor, making the permission change ineffective. A rename forces `ENOENT` on
> the next path lookup, guaranteeing immediate detection.

RS(4,2) tolerates up to **2 simultaneous drives**. With 1 drive offline, 5 of 6
shards survive — well above the minimum `k=4` required for reconstruction.

### Storage overhead for a 100 TB dataset

| Scheme | Data | Total storage | Overhead |
|--------|------|---------------|----------|
| 3× Replication | 100 TB | 300 TB | +200% |
| RS(4,2) | 100 TB | 150 TB | +50% |
| RS(6,2) | 100 TB | 133 TB | +33% |
| RS(10,4) | 100 TB | 140 TB | +40% |

In [ ]:
# Simulate drive1 failure (rename bucket directory)
r = subprocess.run(
    ['docker', 'exec', 'rustfs-server',
     'mv', f'/data/drive1/{BUCKET}', f'/data/drive1/{BUCKET}_FAILED'],
    capture_output=True, text=True,
)
if r.returncode == 0:
    print('SIMULATED: drive1 inaccessible — detected on next I/O')
else:
    print(f'Nothing to rename (drive1 already empty or bucket missing): {r.stderr.strip()}')

In [ ]:
print('Reading objects with drive1 offline — EC should reconstruct transparently:\n')
errors = 0
for key in TEST_OBJECTS:
    try:
        data = s3.get_object(Bucket=BUCKET, Key=key)['Body'].read()
        size = len(data)
        expected = TEST_OBJECTS[key]
        icon = '✅' if size == expected else '❌ wrong size'
        print(f'  [{icon}] {key:30s}  {size:>12,} bytes')
    except Exception as e:
        print(f'  [❌ FAIL] {key}: {e}')
        errors += 1

print(f'\n{"✅ All objects accessible — RS(4,2) reconstructed from 3 surviving drives!" if errors == 0 else f"❌ {errors} failures"}')

In [ ]:
# Restore drive1
r = subprocess.run(
    ['docker', 'exec', 'rustfs-server', 'sh', '-c',
     f'test -d /data/drive1/{BUCKET}_FAILED && '
     f'mv /data/drive1/{BUCKET}_FAILED /data/drive1/{BUCKET} && '
     f'echo "drive1: restored" || echo "drive1: nothing to restore"'],
    capture_output=True, text=True,
)
print(r.stdout.strip())

---
## 6 · Cleanup


In [ ]:
response = s3.list_objects_v2(Bucket=BUCKET)
for obj in response.get('Contents', []):
    s3.delete_object(Bucket=BUCKET, Key=obj['Key'])
    print(f'🗑️  Deleted {obj["Key"]}')

s3.delete_bucket(Bucket=BUCKET)
print(f'🗑️  Deleted bucket: {BUCKET}')

print('\n✅ Cleanup complete!')


---
## 📋 Summary

| Concept | Value |
|---------|-------|
| EC scheme in this lab | RS(4,2) |
| Data shards | 4 |
| Parity shards | 2 |
| Total shards | 6 (across 4 drives) |
| Max drive failures tolerated | 2 |
| Storage efficiency | 67% |

### Key takeaways

- **EC always outperforms replication** in storage efficiency for the same fault tolerance
- The trade-off is **CPU cost**: EC requires encoding on write and reconstruction on repair
- RustFS uses SIMD-accelerated (AVX2) Reed-Solomon to keep encoding overhead minimal
- `ContentLength` in `head_object` shows the **logical** size — parity overhead is hidden

### Next steps

- Read `docs/04-erasure-coding.md` for the full mathematical explanation of Reed-Solomon
- Read `docs/05-fault-tolerance.md` for healing mechanisms and failure scenarios
